# ConvLSTM Large: training and long-horizon evaluation

This experiment trains only the 2.2M-parameter ConvLSTM Large, then evaluates every test trajectory, event-rich cases, all 17 states, and autoregressive horizons up to 1,000 ms. Dataset generation, training, and long rollout print progress and ETA.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile

def project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    work = Path('/kaggle/working')
    if work.exists():
        candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker = candidate / 'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text():
            return candidate
    destination = Path(tempfile.mkdtemp(prefix='hay_scaling_', dir='/kaggle/working'))
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(destination)])
    return destination

ROOT = project_root()
SRC = ROOT / 'src'
assert (SRC / 'hay_single_compartment').is_dir(), f'Package source missing: {SRC}'
sys.path.insert(0, str(SRC))
print('Project:', ROOT)
print('Source:', SRC)

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from hay_single_compartment import INPUT_NAMES, STATE_NAMES, SimulationConfig, generate_dataset, validate_dataset
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_batch, rollout_trajectory, train_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT = Path('/kaggle/working/hay_convlstm_scaling') if Path('/kaggle').exists() else ROOT / 'artifacts' / 'scaling'
OUTPUT.mkdir(parents=True, exist_ok=True)
DATASET = OUTPUT / 'single_compartment_scale.h5'
print('Device:', DEVICE)

## 1. Scale the data before the network

The capacity sweep uses 24 train, 4 validation, and 4 test trajectories of 500 ms. Seeds and complete trajectories remain isolated across splits.

In [ ]:
config = SimulationConfig(
    duration_ms=500.0, warmup_ms=150.0, seed=31415,
    train_trajectories=24, validation_trajectories=4, test_trajectories=4,
)
report = generate_dataset(DATASET, config, progress=True) if not DATASET.exists() else validate_dataset(DATASET)
report

## 2. Train ConvLSTM Large

The Large model uses causal convolutions, three LSTM layers, a 128-step context, cosine learning-rate scheduling, early stopping, and automatic mixed precision on CUDA.

In [ ]:
EXPERIMENTS = [
    dict(run_name='conv_large', hidden_dim=128, layers=3, width_multiplier=2, sequence_length=128, batch_size=32, learning_rate=6e-4, epochs=40),
]

for experiment in EXPERIMENTS:
    probe = build_model(
        'conv_lstm', len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES),
        hidden_dim=experiment['hidden_dim'], layers=experiment['layers'],
        width_multiplier=experiment['width_multiplier'],
    )
    experiment['parameters'] = sum(p.numel() for p in probe.parameters())
pd.DataFrame(EXPERIMENTS)[['run_name', 'parameters', 'sequence_length', 'epochs', 'learning_rate']]

In [ ]:
reports = []
for experiment in EXPERIMENTS:
    print('\nTraining', experiment['run_name'], f"({experiment['parameters']:,} parameters)")
    reports.append(train_model(
        DATASET, OUTPUT / 'models', 'conv_lstm',
        run_name=experiment['run_name'], epochs=experiment['epochs'],
        sequence_length=experiment['sequence_length'], stride=32,
        batch_size=experiment['batch_size'], hidden_dim=experiment['hidden_dim'],
        layers=experiment['layers'], width_multiplier=experiment['width_multiplier'],
        learning_rate=experiment['learning_rate'], patience=8, minimum_epochs=18,
        device=DEVICE, seed=31415, use_amp=True, verbose=True,
    ))
print('Sweep complete')

In [ ]:
comparison = pd.DataFrame([{
    'run': report['run_name'], 'parameters': report['parameters'],
    'epochs_trained': report['epochs_trained'],
    'validation_loss': report['best_validation_loss'],
    'test_voltage_rmse_mV': report['test']['voltage_rmse_mv'],
    'test_normalized_rmse': report['test']['mean_normalized_rmse'],
    'persistence_voltage_rmse_mV': report['test']['persistence_voltage_rmse_mv'],
} for report in reports]).sort_values('validation_loss')
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for report in reports:
    history = pd.DataFrame(report['history'])
    axes[0].plot(history.epoch, history.validation_loss, label=report['run_name'])
    axes[1].plot(history.epoch, history.learning_rate, label=report['run_name'])
axes[0].set(title='Validation', xlabel='epoch', ylabel='weighted loss', yscale='log')
axes[1].set(title='Cosine schedule', xlabel='epoch', ylabel='learning rate')
for axis in axes: axis.grid(alpha=.2); axis.legend()
plt.tight_layout()

## 3. Roll out ConvLSTM Large

The following 200 ms rollout measures compounding error on the first held-out test trajectory.

In [ ]:
winner_name = comparison.iloc[0]['run']
checkpoint = torch.load(OUTPUT / 'models' / f'{winner_name}.pt', map_location=DEVICE, weights_only=False)
winner = build_model(
    checkpoint['architecture'], len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES),
    **checkpoint['model_kwargs'],
).to(DEVICE)
winner.load_state_dict(checkpoint['model_state'])
normalization = Normalization.from_dict(checkpoint['normalization'])
with h5py.File(DATASET, 'r') as h5:
    truth = h5['test/states'][0, :2001]
    future_inputs = h5['test/inputs'][0, :2000]
prediction = rollout_trajectory(winner, truth[0], future_inputs, normalization, DEVICE)
horizons = (1, 5, 10, 25, 50, 100, 150, 200)
rollout_scores = pd.DataFrame([{
    'horizon_ms': horizon,
    'voltage_rmse_mV': float(np.sqrt(np.mean((prediction[:int(horizon/config.dt_ms)+1, 0] - truth[:int(horizon/config.dt_ms)+1, 0])**2))),
} for horizon in horizons])
print('Evaluated model:', winner_name)
rollout_scores

In [ ]:
time_ms = np.arange(len(truth)) * config.dt_ms
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(time_ms, truth[:, 0], label='teacher', lw=1)
axes[0].plot(time_ms, prediction[:, 0], label=winner_name, lw=1, alpha=.8)
axes[0].set_ylabel('V (mV)'); axes[0].legend()
axes[1].plot(time_ms, truth[:, 1] * 1e3, label='teacher')
axes[1].plot(time_ms, prediction[:, 1] * 1e3, label=winner_name, alpha=.8)
axes[1].set(xlabel='time (ms)', ylabel='Ca i (uM)'); axes[1].legend()
plt.tight_layout()

## 4. All-trajectory 1,000 ms stress evaluation

A separate 12-trajectory test bank extends the horizon to 1,000 ms. Batched autoregression evaluates every test trajectory together. Event-rich cases are identified from teacher spike, NMDA, and calcium activity—not from model error.

In [ ]:
CHALLENGE = OUTPUT / 'challenge_1000ms_v1.h5'
challenge_config = SimulationConfig(
    duration_ms=1000.0, warmup_ms=200.0, seed=90421,
    train_trajectories=1, validation_trajectories=1, test_trajectories=12,
)
challenge_report = generate_dataset(CHALLENGE, challenge_config, progress=True) if not CHALLENGE.exists() else validate_dataset(CHALLENGE)
challenge_report

In [ ]:
with h5py.File(CHALLENGE, 'r') as h5:
    challenge_truth = h5['test/states'][...]
    challenge_inputs = h5['test/inputs'][...]
    challenge_spikes = h5['test/spikes'][...]
print('Running', len(challenge_truth), 'batched autoregressive trajectories for 1,000 ms...')
challenge_prediction = rollout_batch(
    winner, challenge_truth[:, 0], challenge_inputs, normalization, DEVICE, progress=True
)
print('Prediction shape:', challenge_prediction.shape)

In [ ]:
long_horizons = (50, 100, 200, 500, 750, 1000)
horizon_rows = []
for trajectory in range(len(challenge_truth)):
    for horizon in long_horizons:
        end = int(horizon / challenge_config.dt_ms) + 1
        error = challenge_prediction[trajectory, :end] - challenge_truth[trajectory, :end]
        normalized_error = error / normalization.state_std
        horizon_rows.append({
            'trajectory': trajectory, 'horizon_ms': horizon,
            'voltage_rmse_mV': float(np.sqrt(np.mean(error[:, 0]**2))),
            'mean_normalized_rmse': float(np.sqrt(np.mean(normalized_error**2, axis=0)).mean()),
        })
horizon_table = pd.DataFrame(horizon_rows)
horizon_summary = horizon_table.groupby('horizon_ms').agg(
    voltage_rmse_mean_mV=('voltage_rmse_mV', 'mean'),
    voltage_rmse_median_mV=('voltage_rmse_mV', 'median'),
    voltage_rmse_worst_mV=('voltage_rmse_mV', 'max'),
    normalized_rmse_mean=('mean_normalized_rmse', 'mean'),
).reset_index()
horizon_summary

In [ ]:
full_error = challenge_prediction - challenge_truth
state_rmse = np.sqrt(np.mean(full_error**2, axis=(0, 1)))
state_units = ['mV', 'mM'] + ['unitless'] * 12 + ['uS'] * 3
state_metrics = pd.DataFrame({
    'state': STATE_NAMES, 'unit': state_units,
    'rmse_physical': state_rmse,
    'rmse_normalized': state_rmse / normalization.state_std,
}).sort_values('rmse_normalized', ascending=False)
state_metrics

In [ ]:
event_inventory = pd.DataFrame({
    'trajectory': np.arange(len(challenge_truth)),
    'spikes': challenge_spikes.sum(axis=1),
    'voltage_peak_mV': challenge_truth[:, :, 0].max(axis=1),
    'calcium_peak_uM': challenge_truth[:, :, 1].max(axis=1) * 1e3,
    'nmda_peak_uS': challenge_truth[:, :, 15].max(axis=1),
})
spiking_ids = event_inventory.loc[event_inventory.spikes > 0, 'trajectory'].tolist()
calcium_ids = event_inventory.nlargest(3, 'calcium_peak_uM').trajectory.tolist()
nmda_ids = event_inventory.nlargest(3, 'nmda_peak_uS').trajectory.tolist()
event_ids = sorted(set(spiking_ids + calcium_ids + nmda_ids))
event_scores = horizon_table[(horizon_table.horizon_ms == 1000) & horizon_table.trajectory.isin(event_ids)]
event_inventory.merge(event_scores, on='trajectory', how='left').sort_values(
    ['spikes', 'calcium_peak_uM', 'nmda_peak_uS'], ascending=False
)

In [ ]:
spike_candidate = int(event_inventory.sort_values(['spikes', 'voltage_peak_mV'], ascending=False).iloc[0].trajectory)
calcium_candidate = int(event_inventory.calcium_peak_uM.idxmax())
nmda_candidate = int(event_inventory.nmda_peak_uS.idxmax())
representatives = list(dict.fromkeys([spike_candidate, calcium_candidate, nmda_candidate]))
challenge_time = np.arange(challenge_truth.shape[1]) * challenge_config.dt_ms
for trajectory in representatives:
    fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
    labels = [('V', 0, 1.0, 'mV'), ('Ca i', 1, 1e3, 'uM'), ('g NMDA', 15, 1.0, 'uS')]
    for axis, (label, state_index, scale, unit) in zip(axes, labels):
        axis.plot(challenge_time, challenge_truth[trajectory, :, state_index] * scale, label='teacher', lw=.9)
        axis.plot(challenge_time, challenge_prediction[trajectory, :, state_index] * scale, label=winner_name, lw=.9, alpha=.8)
        axis.set_ylabel(f'{label} ({unit})'); axis.legend(loc='upper right')
    for spike_index in np.flatnonzero(challenge_spikes[trajectory]):
        axes[0].axvline(spike_index * challenge_config.dt_ms, color='black', alpha=.2, lw=.7)
    axes[-1].set_xlabel('time (ms)')
    fig.suptitle(f'Event-rich test trajectory {trajectory}: spikes={int(challenge_spikes[trajectory].sum())}')
    plt.tight_layout(); plt.show()

## Reading the result

The decisive outputs are the all-trajectory horizon summary, the spike/NMDA/calcium challenge table, and RMSE for all 17 states. Only ConvLSTM Large is trained. The generated HDF5, checkpoint, metrics, and figures remain in `/kaggle/working/hay_convlstm_scaling`.